# Panel statistics

The statistical layer over every full run of one topic: MANOVA of the per-subject distributions on profile columns, PERMANOVA on a sample, distances to the actual result by area, country-level MAE against polling agencies, and the spread of options by area. Nothing is defined here; the code lives in `panel.analysis`, and this notebook only chooses what to feed it.

Until new runs exist, the runs are the legacy ones, keyed by row position and built from the frozen `legacy_clean_data.csv`.

In [ ]:
import pandas as pd

from panel import paths
from panel.analysis import aggregate, compare
from panel.analysis.groups import claimed_by, predicted_by, spread_by
from panel.analysis.plots import comparison, distances, matrices
from panel.analysis.stats import manova, permanova
from panel.dataset import load_claims, load_dataset
from panel.run import ResultStore
from panel.topic import load_topic

In [ ]:
TOPIC_NAME = "election_2021_legacy"

STORE = ResultStore(paths.RESULTS_DIR, TOPIC_NAME)
TOPIC = load_topic(TOPIC_NAME)
OPTIONS = list(TOPIC.options)
TOTAL = TOPIC.reference.total
PROFILE = pd.read_csv(paths.DATA_DIR / "legacy_clean_data.csv", index_col="id")
CLAIMS = load_claims(load_dataset("sod"), TOPIC.name)
reference = TOPIC.load_reference().set_index(TOPIC.reference.key)

# Every full probability run of the topic, keyed by model.
RUNS = {}
for stem in STORE.runs():
    record = STORE.record(stem)
    spec = record["spec"]
    if spec.get("response_model", spec.get("shape")) in ("VotingProbabilities", "VProb") and record["n"] == len(PROFILE):
        RUNS[spec["model"]] = TOPIC.rename(STORE.frame(stem, normalized=True))

# Areas ordered by population (approximate; the ranking is what matters), the whole first.
POPULATION = pd.read_csv(paths.DATASETS_DIR / "regions" / "regions.csv", index_col="region")["population"]
ORDER = [TOTAL] + POPULATION.drop(TOTAL).sort_values(ascending=False).index.tolist()
paths.GRAPHS_DIR.mkdir(exist_ok=True)
print(f"{len(RUNS)} runs: {list(RUNS)}")

## MANOVA

The option probabilities, ILR-transformed, on the profile columns; one report per run, and the effect sizes side by side.

In [ ]:
PREDICTORS = ["gender", "age", "education_level", "income_range", "interest_in_politics", "region", "living_standard"]
reports = {model: manova.manova(PROFILE, frame[OPTIONS], PREDICTORS, OPTIONS) for model, frame in RUNS.items()}
effects = manova.effects(reports)
effects

In [ ]:
fig = distances.effects_bars(effects, "Pillai's trace")
fig = distances.effects_heatmap(effects, "Wilks' lambda")

## PERMANOVA

On a sample: the full 3880 x 3880 distance matrix and its permutations are out of reach. Hellinger distance in probability space; `euclidean_ilr` is the other natural choice.

In [ ]:
SAMPLE = PROFILE.sample(400, random_state=42)
tests = {}
for model, frame in RUNS.items():
    result, report = permanova.panel_permanova(SAMPLE, frame[OPTIONS], PREDICTORS, OPTIONS, distance="hellinger", n_perm=199, seed=42)
    tests[model] = {"F": result.f_value, "p": result.p_value, **report}
pd.DataFrame(tests).T

## Distances to the actual result, by area

Each run aggregated per region (the whole country first), the claimed votes as one more source, and the legacy latent tables where they exist. The dashed line is the first source's value for the whole.

In [ ]:
source_tables = {}
for model, frame in RUNS.items():
    by_area = predicted_by(frame, PROFILE, "region", TOPIC, total=TOTAL).loc[ORDER]
    source_tables[model] = compare.distance_table(reference.loc[ORDER], by_area, OPTIONS)
claimed = compare.smooth(claimed_by(PROFILE, "region", CLAIMS, TOPIC, total=TOTAL).loc[ORDER], OPTIONS)
source_tables["claimed"] = compare.distance_table(reference.loc[ORDER], claimed, OPTIONS)
if STORE.path("preferences").exists():
    for path in sorted(STORE.path("preferences").glob("by_region_latent_*.csv")):
        table = TOPIC.rename(pd.read_csv(path, index_col=0)).set_index("region").loc[ORDER]
        source_tables[path.stem[len("by_region_"):]] = compare.distance_table(reference.loc[ORDER], table, OPTIONS)
distances.distance_lines(source_tables, ORDER, ["MAE", "KL Divergence"], total=TOTAL, ncols=1, height_per_row=500, figure_width=1200)

In [ ]:
distances.distance_bars(source_tables, TOTAL, ["MAE", "KL Divergence"], figure_width=1200)

## Country-level MAE by source, polling agencies included

`data/agentury.csv` holds the agencies' final polls in the topic's native names; `Topic.rename` reads them as ids.

In [ ]:
country = {model: aggregate.aggregate_results_from_df(frame, TOPIC) for model, frame in RUNS.items()}
country["claimed"] = aggregate.claimed_distribution(PROFILE, CLAIMS, TOPIC)
agencies = TOPIC.rename(pd.read_csv(paths.DATA_DIR / "agentury.csv", index_col=0, delimiter=";", encoding="utf-8-sig").dropna()).drop(columns=["url"])
for agency, poll in agencies.iterrows():
    country[agency] = poll
actual = reference.loc[TOTAL, OPTIONS]
mae = pd.Series({source: compare.distances(actual, series[OPTIONS])["MAE"] for source, series in country.items()}).sort_values()
fig = comparison.bars_by_source(mae, "MAE (%)", scale=100, title="Country-level MAE against the actual result")
fig.savefig(paths.GRAPHS_DIR / f"country_mae_by_source_{TOPIC_NAME}.png", bbox_inches="tight")
mae.round(4)

## Spread of options by area

Standard deviation of each option's probability within each region, per run.

In [ ]:
spreads = {model: spread_by(frame.join(PROFILE[["region"]]), "region", OPTIONS)[ORDER[1:]] for model, frame in RUNS.items()}
first = next(iter(spreads))
fig = matrices.spread_heatmap(spreads[first], title=f"Spread of options by region - {first}", labels=TOPIC.names)
fig = matrices.spread_heatmap_grid(spreads, ncols=2, per_subplot_size=(6, 5), title="Spread of options by region, every run")